In [3]:
# ============================================================
# TRANSFORM RESULTS — LOCAL SILVER LAYER
# ============================================================

import os
from pyspark.sql import functions as F

In [4]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG
# -----------------------------------------
try:
    BRONZE_ROOT
except NameError:
    import nbformat
    from nbconvert import PythonExporter

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    config_path = os.path.join(project_root, "/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 23:08:18 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/08 23:08:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 23:08:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/08 23:08:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/08 23:08:18 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/08 23:08:18 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/08 23:08:18 

✔ Spark Session Initialized
✔ Environment Config Loaded
PROJECT_ROOT: /Users/manoharazalki/F1-ANALYTICS


In [5]:
# -----------------------------------------
# 2. Define Bronze + Silver paths
# -----------------------------------------
bronze_file = f"{BRONZE_ROOT}/results/results.parquet"
silver_folder = f"{SILVER_ROOT}/results"
silver_file = f"{silver_folder}/results_silver.parquet"

os.makedirs(silver_folder, exist_ok=True)

print("Bronze:", bronze_file)
print("Silver:", silver_file)

Bronze: /Users/manoharazalki/F1-ANALYTICS/data/bronze/results/results.parquet
Silver: /Users/manoharazalki/F1-ANALYTICS/data/silver/results/results_silver.parquet


In [6]:
# -----------------------------------------
# 3. Read Bronze Results
# -----------------------------------------
results_df = spark.read.parquet(bronze_file)
print("Bronze results loaded")
results_df.show(5, truncate=False)

Bronze results loaded
+----------+--------------------+-----+------+-------------------------------------------------------+-------------+------------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|date      |raceName            |round|season|url                                                    |constructorId|driverId          |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                  |
+----------+--------------------+-----+------+-------------------------------------------------------+-------------+------------------+----+----+------+------+--------+------------+--------+--------------------------+-------------------------------------------------------------+
|1995-03-26|brazilian grand prix|1    |1995  |https://en.wikipedia.org/wiki/1995_Brazilian_Grand_Prix|benetton     |michael_schumacher|2  

In [7]:
# -----------------------------------------
# 4. Select + rename columns (standardization)
# -----------------------------------------
results_selected_df = (
    results_df
        .select(
            "season",
            "round",
            "constructorId",
            "driverId",
            "date",
            "raceName",
            "grid",
            "laps",
            "number",
            "points",
            "position",
            "positionText",
            "status",
            "ingest_timestamp",
            "source_file"
        )
        .withColumnsRenamed(
            {
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "car_number",
                "position": "final_position",
                "positionText": "finish_position_text"
            }
        )
)

print("✔ Results renamed preview")
results_selected_df.show(5, truncate=False)

✔ Results renamed preview
+------+-----+--------------+------------------+----------+--------------------+-------------+--------------+----------+------+--------------+--------------------+--------+--------------------------+-------------------------------------------------------------+
|season|round|constructor_id|driver_id         |race_date |race_name           |grid_position|completed_laps|car_number|points|final_position|finish_position_text|status  |ingest_timestamp          |source_file                                                  |
+------+-----+--------------+------------------+----------+--------------------+-------------+--------------+----------+------+--------------+--------------------+--------+--------------------------+-------------------------------------------------------------+
|1995  |1    |benetton      |michael_schumacher|1995-03-26|brazilian grand prix|2            |71            |1         |10.0  |1             |1                   |Finished|2026-06-08 23:08

In [8]:
# -----------------------------------------
# 5. Business key validation
# -----------------------------------------
results_valid_df = (
    results_selected_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

print("Invalid rows removed:", results_selected_df.count() - results_valid_df.count())

Invalid rows removed: 108


In [9]:
# -----------------------------------------
# 6. Title Case race_name
# -----------------------------------------
results_final_df = (
    results_valid_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

print("Results Silver preview")
results_final_df.show(5, truncate=False)

Results Silver preview


26/06/08 23:08:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+-----+--------------+--------------+----------+------------------+-------------+--------------+----------+------+--------------+--------------------+------------+--------------------------+-------------------------------------------------------------+
|season|round|constructor_id|driver_id     |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|finish_position_text|status      |ingest_timestamp          |source_file                                                  |
+------+-----+--------------+--------------+----------+------------------+-------------+--------------+----------+------+--------------+--------------------+------------+--------------------------+-------------------------------------------------------------+
|1950  |1    |era           |harrison      |1950-05-13|British Grand Prix|15           |67            |11        |0.0   |7             |7                   |+3 Laps     |2026-06-08 23:08:03.439846|/Users/manoharazalki/F1

In [10]:
# -----------------------------------------
# 7. Write Silver output (Option C)
# -----------------------------------------
(
    results_final_df
        .write
        .mode("overwrite")
        .parquet(silver_file)
)

print(f"Results Silver written to: {silver_file}")

Results Silver written to: /Users/manoharazalki/F1-ANALYTICS/data/silver/results/results_silver.parquet


In [11]:
# -----------------------------------------
# 8. Read back for validation
# -----------------------------------------
spark.read.parquet(silver_file).show(5, truncate=False)

+------+-----+--------------+-----------+----------+------------------+-------------+--------------+----------+------+--------------+--------------------+------------+--------------------------+-------------------------------------------------------------+
|season|round|constructor_id|driver_id  |race_date |race_name         |grid_position|completed_laps|car_number|points|final_position|finish_position_text|status      |ingest_timestamp          |source_file                                                  |
+------+-----+--------------+-----------+----------+------------------+-------------+--------------+----------+------+--------------+--------------------+------------+--------------------------+-------------------------------------------------------------+
|1950  |1    |alfa          |reg_parnell|1950-05-13|British Grand Prix|4            |70            |4         |4.0   |3             |3                   |Finished    |2026-06-08 23:08:03.439846|/Users/manoharazalki/F1-ANALYTICS/d